# EDA 09 — Sante Environnementale (Espaces Verts + Arbres + Stations Air)
**Sources** : Airparif ArcGIS | Paris Open Data Ilots de fraicheur | Paris Open Data Arbres

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "09_health_env"
draw_schema(
    "Bronze Schema — Stations de Mesure Airparif",
    [
        ("station_id",    "str",   "Identifiant station ArcGIS"),
        ("station_name",  "str",   "Nom de la station"),
        ("station_type",  "str",   "Type : fond | trafic | industriel"),
        ("latitude",      "float", "Latitude"),
        ("longitude",     "float", "Longitude"),
        ("commune_code",  "str",   "Code INSEE commune"),
        ("arrondissement","int",   "Arrondissement (1-20)"),
        ("polluants",     "str",   "Liste JSON des polluants mesures"),
        ("ingested_at",   "datetime","Horodatage UTC"),
    ], PREFIX, "schema_stations"
)

In [ ]:
draw_schema(
    "Bronze Schema — Ilots de Fraicheur (Paris Open Data)",
    [
        ("site_id",       "str",   "Identifiant Paris OD"),
        ("nom",           "str",   "Nom du site"),
        ("categorie",     "str",   "Categorie : jardin | square | bois | promenade | parc"),
        ("surface_ha",    "float", "Surface en hectares"),
        ("adresse",       "str",   "Adresse"),
        ("arrondissement","int",   "Arrondissement"),
        ("latitude",      "float", "Latitude du centroide"),
        ("longitude",     "float", "Longitude du centroide"),
        ("ingested_at",   "datetime","Horodatage UTC"),
    ], PREFIX, "schema_ilots"
)

In [ ]:
draw_schema(
    "Bronze Schema — Arbres de Paris (Canopee urbaine)",
    [
        ("arbre_id",         "str",   "Identifiant arbre"),
        ("genre",            "str",   "Genre botanique (Platanus, Acer, Quercus...)"),
        ("espece",           "str",   "Espece botanique"),
        ("libelle_francais", "str",   "Nom commun francais"),
        ("hauteur_m",        "float", "Hauteur estimee (m)"),
        ("circonference_cm", "int",   "Circonference du tronc (cm)"),
        ("annee_plantation", "int",   "Annee de plantation"),
        ("arrondissement",   "int",   "Arrondissement"),
        ("latitude",         "float", "Latitude"),
        ("longitude",        "float", "Longitude"),
        ("ingested_at",      "datetime","Horodatage UTC"),
    ], PREFIX, "schema_arbres"
)

In [ ]:
def load_latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()
df_sta = load_latest(f"{BRONZE_ROOT}/airparif_stations/**/*.parquet")
df_ilo = load_latest(f"{BRONZE_ROOT}/paris_ilots_fraicheur/**/*.parquet")
df_arb = load_latest(f"{BRONZE_ROOT}/paris_canopee/**/*.parquet")
rng = np.random.default_rng(42)
if df_sta.empty:
    df_sta = pd.DataFrame([{"station_id":f"STA{i:03d}","station_name":f"Station {i}",
        "station_type":rng.choice(["fond","trafic","industriel"],p=[0.5,0.4,0.1]),
        "latitude":48.8566+rng.uniform(-0.07,0.07),"longitude":2.3522+rng.uniform(-0.08,0.08),
        "commune_code":f"751{rng.integers(1,21):02d}","arrondissement":int(rng.integers(1,21)),
        "polluants":'["NO2","PM2.5","O3"]'} for i in range(20)])
if df_ilo.empty:
    cats = ["jardin","square","bois","promenade","parc"]
    df_ilo = pd.DataFrame([{"site_id":f"ILO{i:04d}","nom":f"Espace vert {i}","categorie":rng.choice(cats),
        "surface_ha":rng.exponential(1.5)+0.1,"adresse":"Paris","arrondissement":int(rng.integers(1,21)),
        "latitude":48.8566+rng.uniform(-0.07,0.07),"longitude":2.3522+rng.uniform(-0.08,0.08)} for i in range(180)])
if df_arb.empty:
    genres = ["Platanus","Acer","Quercus","Tilia","Prunus","Fraxinus"]
    df_arb = pd.DataFrame([{"arbre_id":f"ARB{i:06d}","genre":rng.choice(genres),
        "hauteur_m":rng.exponential(9)+5,"circonference_cm":int(rng.exponential(55)+20),
        "annee_plantation":int(rng.integers(1900,2024)),"arrondissement":int(rng.integers(1,21)),
        "latitude":48.8566+rng.uniform(-0.07,0.07),"longitude":2.3522+rng.uniform(-0.08,0.08)} for i in range(5000)])
print(f"Stations {df_sta.shape} | Ilots {df_ilo.shape} | Arbres {df_arb.shape}")

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,6))
arr_ilo = df_ilo.groupby("arrondissement").agg(surface=("surface_ha","sum"))
norm_s = (arr_ilo["surface"]-arr_ilo["surface"].min())/(arr_ilo["surface"].max()-arr_ilo["surface"].min())
s_sorted = arr_ilo.sort_values("surface",ascending=False)
axes[0].bar(s_sorted.index.astype(str),s_sorted["surface"],color=plt.cm.Greens(0.25+norm_s.reindex(s_sorted.index)*0.7),edgecolor="white")
axes[0].set_xlabel("Arrondissement"); axes[0].set_ylabel("Surface (ha)"); axes[0].set_title("Surface d'espaces verts par arrondissement")
plt.setp(axes[0].xaxis.get_majorticklabels(),rotation=45)
arr_arb = df_arb.groupby("arrondissement").size().sort_values(ascending=False)
norm_a = (arr_arb-arr_arb.min())/(arr_arb.max()-arr_arb.min())
axes[1].bar(arr_arb.index.astype(str),arr_arb.values,color=plt.cm.YlGn(0.25+norm_a*0.7),edgecolor="white")
axes[1].set_xlabel("Arrondissement"); axes[1].set_ylabel("Nombre d'arbres"); axes[1].set_title("Densite arboree (canopee) par arrondissement")
plt.setp(axes[1].xaxis.get_majorticklabels(),rotation=45)
save_fig("espaces_verts_et_arbres", PREFIX); plt.show()

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,6))
genres_top = df_arb["genre"].value_counts().head(8)
axes[0].barh(genres_top.index[::-1],genres_top.values[::-1],color=plt.cm.YlGn(np.linspace(0.4,0.9,len(genres_top))),edgecolor="white")
axes[0].set_xlabel("Nombre d'arbres"); axes[0].set_title("Top 8 genres botaniques — Paris")
cat_counts = df_ilo["categorie"].value_counts()
axes[1].pie(cat_counts.values,labels=cat_counts.index,autopct="%1.1f%%",startangle=90,
            colors=plt.cm.Greens(np.linspace(0.35,0.85,len(cat_counts))),
            wedgeprops=dict(edgecolor="white",linewidth=1.5))
axes[1].set_title("Categories d'espaces de fraicheur")
save_fig("genres_et_categories", PREFIX); plt.show()